# STEP 2 - Competitive Landscape

| Question to answer (Presentation deck, Step 2) | How we measure it in this notebook |
|---|---|
| How concentrated is the field, and is it consolidating? | Six measures of concentration from the literature (Sections 3–4); the Herfindahl index and the 4-firm ratio over time (Section 5.1) |
| Who are the incumbents, and who are the new entrants? | Top applicants with type and country; new applicants per year, their share of families, and the rate of disappearance (Section 5.2) |
| Which subfields are growing fastest? | Growth of the CPC groups of the sector between two periods (Section 5.3) |
| Which political, economic, social and technological forces shape it? | PEST analysis of the political, economic, social and technological forces, with sources outside PATSTAT (Section 5.4) |

**Contents**

1. Setup and configuration
2. The unit of analysis: applicants and harmonisation, i.e. Entity Linking
3. Proxies for concentration: six measures from the literature
4. Example: inventive concentration in fuel cells (Liston-Heyes & Pilkington, 2004)
5. Answering the Step 2 questions for our sector

## 1. Setup and configuration

In [1]:
%pip install -q cleanco rapidfuzz # cleanco: legal forms (method B); rapidfuzz: fuzzy matching of misspelled legal forms

Note: you may need to restart the kernel to use updated packages.


In [14]:
import re
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from scipy import stats
from sqlalchemy import func, or_, and_
from cleanco import basename
from rapidfuzz import fuzz

# ---------------------------------------------------------------- environment & config
REPO = Path.cwd() if (Path.cwd() / "config").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

from stage00_config import load_config
cfg = load_config()

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR,
    TLS206_PERSON, TLS207_PERS_APPLN, TLS209_APPLN_IPC, 
    TLS211_PAT_PUBLN, TLS224_APPLN_CPC
)

ENV = cfg.env
OUTPUT_DIR =  Path("outputs/step2")  
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- applicant names (Section 2)
NAME_METHOD = "cleanco"           # "psn" (PATSTAT names as they are), "dictionary" or "cleanco" or "terrorizer"
GOLD_STANDARD_FILE = "PATSTAT_500_label_datav4.csv"   # hand-labelled names (Section 2.5), in the same folder as this notebook

# ---------------------------------------------------------------- replication (Section 4): the paper's design
#REPL_OFFICE = "US"                                      # USPTO patents...
#REPL_GRANTED = (date(1976, 1, 1), date(2002, 12, 31))   # ...granted between these two dates
#REPL_KEYWORDS = ("fuel cell",)                          # searched in titles and abstracts
#REPL_CLASSES = ("H01M8",)                               # IPC/CPC H01M 8: fuel cells
#REPL_SELECTION = "keywords or classes"                  # "keywords", "classes" or "keywords or classes"
#PAPER_COUNTRIES = ("AU", "CA", "FR", "DE", "IL", "IT", "JP", "NL", "SE", "CH", "GB", "US")


# ---------------------------------------------------------------- sector parameters
IPR_TYPE = "PI"                      # patents of invention only
SECTOR_NAME = cfg.sector_name
SPLIT_YEAR = 2015                    # incumbents file before SPLIT_YEAR, entrants from SPLIT_YEAR on
WINDOW = 3                           # years per window when following concentration over time
TOP_N = 30                           # rows in the tables of top applicants, i.e number of players selected

COUNTRY_NAMES = {"AU": "Australia", "CA": "Canada", "FR": "France", "DE": "Germany", "IL": "Israel", "IT": "Italy",
                 "JP": "Japan", "NL": "Netherlands", "SE": "Sweden", "CH": "Switzerland", "GB": "UK", "US": "USA", "CN": "China", "KR": "South Korea"}

patstat = PatstatClient(env=ENV)
db = patstat.orm()
print(f"Sector: {SECTOR_NAME} | env: {ENV}")

Sector: Quantum computing | env: PROD


## 2. The unit of analysis: applicants

In Step 1 we counted **families** and **applications**. Concentration needs one more unit: the **applicant**, the owner of the patent. We must decide when two names on two patents are **the same applicant**.

In [15]:
def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def keyword_condition(terms, exclusions=()):
    """Any term appears in the title or abstract and no exclusion phrase does (case-insensitive).

    A missing title or abstract is read as empty text: NOT applied to a NULL comparison is NULL,
    which would drop the application silently.
    """
    title = func.lower(func.coalesce(TLS202_APPLN_TITLE.appln_title, ""))
    abstract = func.lower(func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""))
    match = or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))
    if not exclusions:
        return match
    excluded = or_(*(col.like(f"%{x.lower()}%") for x in exclusions for col in (title, abstract)))
    return and_(match, ~excluded)

                          
family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
by_cpc = (db.query(family_id)
          .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
          .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, cfg.cpc_prefixes)))

by_kw = (db.query(family_id)
         .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
         .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
         .filter(keyword_condition(cfg.keywords)))

sector_ids = by_cpc.union(by_kw)

# Prepare the subquery for families within year_min and year_max
family_year = func.min(TLS201_APPLN.earliest_filing_year)
families = (
    db.query(TLS201_APPLN.docdb_family_id.label("unit_id"), family_year.label("year"))
    .filter(TLS201_APPLN.ipr_type == cfg.ipr_type,
            TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select()))
    .group_by(TLS201_APPLN.docdb_family_id)
    .having(family_year.between(cfg.year_min, cfg.year_max))
    .subquery()
)

# Extract all the applicant names for these families
q_sector = (
    db.query(families.c.unit_id, families.c.year, TLS206_PERSON.psn_id, TLS206_PERSON.psn_name,
             TLS206_PERSON.psn_sector, func.trim(TLS206_PERSON.person_ctry_code).label("ctry"))
    .select_from(families)
    .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == families.c.unit_id)
    .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
    .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
    .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)
    .distinct()
)
raw_sector = patstat.df(q_sector)
print(f"Extracted {raw_sector['unit_id'].nunique():,} patent families with their respective applicants.")

Extracted 12,280 patent families with their respective applicants.


### 2.1 The "name game"

The same company appears in PATSTAT under many spellings (*SIEMENS AG*, *Siemens Aktiengesellschaft*, *SIEMENS AKTIENGESELLSCHAFT MUENCHEN*…). If we count each spelling as a different firm, a large firm is split into many small ones and **every concentration measure is wrong**. `tls206_person` offers:

| Column | What it is |
|---|---|
| `person_name` | the name exactly as printed on the document: never use it to count applicants |
| `psn_id`, `psn_name` | the **PATSTAT Standardised Name** (EPO and K.U. Leuven): a first harmonisation, available for every person |
| `han_id`, `han_name` | the **OECD HAN** (Harmonised Applicant Names): a stronger harmonisation of firm names |

`psn_name` is a good start, but it still leaves variants (*TOYOTA JIDOSHA KK* and *TOYOTA MOTOR CORP*, with and without legal forms…), and it never merges a subsidiary with its parent.

### 2.2 Three possible ways to consolidate names further

| Method | How it works | Good at | Misses |
|---|---|---|---|
| **A · Dictionary of legal forms** | upper-case the name, remove punctuation and the legal-form words (INC, LTD, GMBH, AG, KK…) | variants that differ only in punctuation or legal form | translations, abbreviations, groups |
| **B · `cleanco`** | a Python package with the legal forms ("business entity" extensions) of many countries | the same, also abbreviations with dots (S.p.A., N.V.) | the same |
| **C · Terrorizer** | NLP and network techniques: name similarity, a knowledge base of company names, community detection on the network of similar names (https://github.com/graziasveva/Terrorizer) | spelling variants, abbreviations, names in different forms | it needs a check of the groups it creates |

Whatever the method, **look at the top of your list by hand**: in the end, you decide what one applicant is.

**Method A** is a few lines of code: a set of legal-form words, and a function that removes them from the beginning and the end of a name.

In [16]:
# Legal-form words to remove (extend the set for the countries of your sector)
LEGAL_FORMS = {
    "INC", "INCORPORATED", "CORP", "CORPORATION", "CO", "COMPANY", "LTD", "LIMITED", "LLC", "LP", "PLC",
    "GMBH", "AG", "KG", "MBH", "SE", "SA", "SAS", "SPA", "SRL", "BV", "NV", "AB", "AS", "ASA", "OY", "OYJ",
    "KK", "KABUSHIKI", "KAISHA", "LTDA", "PTY", "THE", "AKTIENGESELLSCHAFT", "GESELLSCHAFT",
    "GCV", "CV", "SARL", "SRO", "PTE", "BHD", "OOO", "ZAO", "OAO", "PJSC", "JSC", "LLP", "ULC", "UG", "SL"}


def clean_with_dictionary(name):
    "Upper-case, keep letters and digits only, and drop legal-form words at the start and at the end."
    words = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper()).split()
    while words and words[-1] in LEGAL_FORMS:      # e.g. "TOYOTA JIDOSHA KABUSHIKI KAISHA" -> "TOYOTA JIDOSHA"
        words.pop()
    while words and words[0] in LEGAL_FORMS:       # e.g. "KABUSHIKI KAISHA TOSHIBA" -> "TOSHIBA"
        words.pop(0)
    return " ".join(words)


examples = ["SIEMENS AG", "Siemens Aktiengesellschaft", "BALLARD POWER SYSTEMS INC.", "Ballard Power Systems",
            "TOYOTA JIDOSHA KABUSHIKI KAISHA", "KABUSHIKI KAISHA TOSHIBA", "Toyota Motor Corp",
            "FIAT S.p.A.", "Umicore N.V."]
pd.DataFrame({"name": examples, "method A": [clean_with_dictionary(n) for n in examples]})

,name,method A
0,SIEMENS AG,SIEMENS
1,Siemens Aktiengesellschaft,SIEMENS
2,BALLARD POWER SYSTEMS INC.,BALLARD POWER SYSTEMS
3,Ballard Power Systems,BALLARD POWER SYSTEMS
4,TOYOTA JIDOSHA KABUSHIKI KAISHA,TOYOTA JIDOSHA
5,KABUSHIKI KAISHA TOSHIBA,TOSHIBA
6,Toyota Motor Corp,TOYOTA MOTOR
7,FIAT S.p.A.,FIAT S P A
8,Umicore N.V.,UMICORE N V


**Method B** uses the `cleanco` package, which knows the legal forms of many countries, also when they are written as abbreviations with dots: method A turns *FIAT S.p.A.* into *FIAT S P A*, cleanco into *FIAT*. We use cleanco first, then the cleaning of method A.

In [17]:
def clean_with_cleanco(name):
    "Remove legal forms with cleanco, then apply the same upper-case cleaning as method A."
    return clean_with_dictionary(basename(str(name)))


pd.DataFrame({"name": examples,
              "method A": [clean_with_dictionary(n) for n in examples],
              "method B": [clean_with_cleanco(n) for n in examples]})

,name,method A,method B
0,SIEMENS AG,SIEMENS,SIEMENS
1,Siemens Aktiengesellschaft,SIEMENS,SIEMENS
2,BALLARD POWER SYSTEMS INC.,BALLARD POWER SYSTEMS,BALLARD POWER SYSTEMS
3,Ballard Power Systems,BALLARD POWER SYSTEMS,BALLARD POWER SYSTEMS
4,TOYOTA JIDOSHA KABUSHIKI KAISHA,TOYOTA JIDOSHA,TOYOTA JIDOSHA
5,KABUSHIKI KAISHA TOSHIBA,TOSHIBA,TOSHIBA
6,Toyota Motor Corp,TOYOTA MOTOR,TOYOTA MOTOR
7,FIAT S.p.A.,FIAT S P A,FIAT
8,Umicore N.V.,UMICORE N V,UMICORE


**Method C** uses NLP and network techniques (such as name similarity and community detection on networks of similar names, via Terrorizer https://github.com/graziasveva/Terrorizer) to handle spelling variants, abbreviations, and names written in entirely different forms. While powerful for merging complex variants, it always requires a manual check of the generated groups.

We will reproduce part of its ideas with `rapidfuzz` (similarity between names) and with the linking of subsidiaries to the brand (same first word).

In [18]:
%pip install --upgrade terrorizer
from terrorizer import terrorizer_disambiguator

ERROR: Could not find a version that satisfies the requirement terrorizer (from versions: none)
ERROR: No matching distribution found for terrorizer
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'terrorizer'

In [ ]:
df = pd.DataFrame({
    "name": examples,
    "method A": [clean_with_dictionary(n) for n in examples],
    "method B": [clean_with_cleanco(n) for n in examples]
})

try:
    results = terrorizer_disambiguator(df, column_name='method B')
    # If the function returns a dataframe, we extract the disambiguated column
    df["method C"] = results.iloc[:, -1] 
except Exception as e:
    # Fallback placeholder in case the package fails to load or requires a knowledge base file
    df["method C"] = df["method B"]
    print(f"Terrorizer execution note: {e}")

df

### 2.3 From persons to applicants (`TLS206_PERSON`): our harmonisation

Our name harmonisation combines methods A and B with sector-specific steps, applied in this order:

1. **Methods A and B**: upper-case, accent removal, and removal of legal forms with our dictionary and with `cleanco`.
2. **Specific cleaning**: institutional wrappers (e.g. *Board of Trustees of…*), abbreviations and translations (*Univ*, *Université* → UNIVERSITY), filler words, word order of universities.
3. **Fuzzy matching of legal forms (`rapidfuzz`)**: a final word that is a typo of a long legal form (e.g. *COPPORATION*, *CORPAORATION*) is removed when its similarity to the form (`fuzz.ratio`) is at least 85 and its length differs by at most one character.
4. **Manual aliases** for the main players of the sector (e.g. IBM, Google, Microsoft, D-Wave), decided after a manual check of the top applicants.
5. **Linking subsidiaries to the brand**: a name that starts, as whole words, with the name of another applicant in the list (e.g. *NOKIA DATA SYSTEMS* → *NOKIA*) is linked to it. Universities, individuals and generic names never act as anchors.

From the harmonised names we derive:

- **`applicant`**: the consolidated name;
- **`type`**: firm, university, individual… from `psn_sector`, or inferred from the name when `psn_sector` is empty;
- **`ctry`**: the applicant's country. A firm can have different countries on different patents (or none): we give each applicant the country that appears **most often** among its patents.

In [19]:
# ---------------------------------------------------------------- typos in legal forms
LONG_LEGAL = ["CORPORATION", "INCORPORATED", "LIMITED", "COMPANY", "AKTIENGESELLSCHAFT", "GESELLSCHAFT", "KABUSHIKI"]

def strip_misspelled_legal_forms(name):
    "Drop a final word that is a typo of a long legal form (COPPORATION, CORPAORATION...)."
    words = name.split()
    while len(words) > 1 and any(abs(len(words[-1]) - len(f)) <= 1 and fuzz.ratio(words[-1], f) >= 85
                                 for f in LONG_LEGAL):
        words.pop()
    return " ".join(words)

# ---------------------------------------------------------------- link subsidiaries to the brand ("anchor")
GENERIC_ANCHORS = {   # words that must never absorb other names: add the generic words of our sector
    "UNIVERSITY", "INSTITUTE", "NATIONAL", "INTERNATIONAL", "GENERAL", "UNITED", "AMERICAN", "CHINA", "CHINESE",
    "BEIJING", "SHANGHAI", "SHENZHEN", "TECHNOLOGY", "SYSTEMS", "QUANTUM", "SCIENCE", "RESEARCH", "ELECTRONICS",
    "ELECTRIC", "COMPUTING", "SEMICONDUCTOR", "LABORATORY", "CENTER", "FOUNDATION", "GLOBAL", "DIGITAL",
    "ADVANCED", "APPLIED", "FIRST", "SOUTH", "NORTH", "PHOTONIC", "PHOTONICS", "SILICON", "NANO"}

def link_subsidiaries(mapping, skip=frozenset(), min_len=5):
    """{name: harmonised} -> {name: harmonised or its anchor}. A name that starts, as whole words, with another
    harmonised name of the list is linked to it: NOKIA ALUMIINI -> NOKIA. NOKIAN TYRES stays apart.
    Never anchors: universities (NANJING UNIVERSITY != NANJING UNIVERSITY OF POSTS), individuals,
    and names made only of generic words (QUANTUM TECHNOLOGY)."""
    anchors = {v for v in set(mapping.values())
               if v and len(v) >= min_len and v not in skip
               and not v.startswith("UNIVERSITY")
               and not all(w in GENERIC_ANCHORS for w in v.split())}
    def link(v):
        words = v.split()
        for k in range(1, len(words)):                 # shortest prefix first = the brand
            prefix = " ".join(words[:k])
            if prefix in anchors:
                return prefix
        return v
    return {n: link(v) for n, v in mapping.items()}

In [20]:
import unicodedata

# ---------------------------------------------------------------- specific cleaning
ABBREVIATIONS = {   # same organisation, different spelling or language
    "UNIV": "UNIVERSITY", "UNIVERSITE": "UNIVERSITY", "UNIVERSITAT": "UNIVERSITY", "UNIVERSITAET": "UNIVERSITY",
    "UNIVERSITA": "UNIVERSITY", "UNIVERSIDAD": "UNIVERSITY", "UNIVERSIDADE": "UNIVERSITY", "UNIVERSITEIT": "UNIVERSITY",
    "INST": "INSTITUTE", "INSTITUT": "INSTITUTE", "INSTITUTO": "INSTITUTE", "ISTITUTO": "INSTITUTE",
    "NATL": "NATIONAL", "NAT": "NATIONAL", "LAB": "LABORATORY", "LABS": "LABORATORIES", "TECH": "TECHNOLOGY",
    "TECHNOLOGIES": "TECHNOLOGY", "INTL": "INTERNATIONAL", "CENT": "CENTER", "CENTRE": "CENTER", "RES": "RESEARCH",
    "SYST": "SYSTEMS", "SCI": "SCIENCE", "SCIENCES": "SCIENCE", "ELECTR": "ELECTRONICS",
}
FILLER_WORDS = {"OF", "THE", "AND", "DE", "DEL", "DELLA", "DI", "DU", "DES", "LA", "LE", "DER", "DIE", "DAS",
                "FUR", "FUER", "FOR", "AT", "IN"}
TRAILING_NOISE = {"HOLDING", "HOLDINGS", "GROUP", "IP"}
NAME_PREFIXES = [   # institutional wrappers that hide the real owner
    r"^(THE )?BOARD OF TRUSTEES OF (THE )?", r"^(THE )?BOARD OF REGENTS( OF)?( THE)? ", r"^(THE )?REGENTS OF (THE )?",
    r"^(THE )?TRUSTEES OF (THE )?", r"^PRESIDENT AND FELLOWS OF ", r"^THE GOVERNING COUNCIL OF (THE )?",
    r"^UNITED STATES OF AMERICA AS REPRESENTED BY (THE )?", r"^(THE )?GOVERNMENT OF ",
]
# Manual aliases for the big players of the sector: (regex on the cleaned name, canonical name)
ORG_ALIASES = [
    (r"^INTERNATIONAL BUSINESS MACHINES|^IBM\b",       "IBM"),
    (r"^GOOGLE\b",                                      "GOOGLE"),
    (r"^MICROSOFT\b",                                   "MICROSOFT"),
    (r"^D WAVE\b|^DWAVE\b",                             "D-WAVE SYSTEMS"),
    (r"^HEWLETT PACKARD",                               "HEWLETT PACKARD"),
    (r"^NIPPON TELEGRAPH|^NTT\b",                       "NTT"),
    (r"^ORIGIN QUANTUM",                                "ORIGIN QUANTUM"),
    (r"^CENTER NATIONAL RECHERCHE|^CENTRE NATIONAL RECHERCHE|CNRS", "CNRS"),
    (r"^COMMISSARIAT ENERGIE ATOMIQUE",                 "CEA"),
    (r"^ELECTRONIC SCIENCE RESEARCH INSTITUTE",         "ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA"),
    (r"^INFORMATION SCIENCE ACADEMY",                   "INFORMATION SCIENCE ACADEMY CHINA"),
    (r"^BEIJING BAIDU NETCOM|^BAIDU\b",                    "BAIDU"),
    (r"^SGCC\b|^STATE GRID CORPORATION|^STATE GRID CORP",  "STATE GRID CORPORATION OF CHINA"),
    (r"^MIT\b|^MASSACHUSETTS INSTITUTE TECHNOLOGY",        "MIT"),
    (r"^ETRI\b|^ELECTRONICS TELECOMMUNICATIONS RESEARCH INSTITUTE", "ETRI"),
    (r"^HUAWEI",                                           "HUAWEI"),
    (r"^QUANTUMCTEK|^QUANTUM CTEK",                        "QUANTUMCTEK"),
    (r"^RIGETTI",                                          "RIGETTI"),
    (r"^QUANTINUUM",                                       "QUANTINUUM"),
    (r"^IONQ",                                             "IONQ"),
    (r"^INTEL\b",                                          "INTEL"),
    (r"^TOSHIBA\b",                                        "TOSHIBA"),
    (r"^UNIVERSITY OXFORD\b",                              "UNIVERSITY OXFORD"),
]

_ORG = [(re.compile(p), c) for p, c in ORG_ALIASES]


def ascii_upper(name):
    "Upper-case and remove accents (Müller -> MULLER, Université -> UNIVERSITE)."
    s = str(name).replace("ß", "ss").replace("Ø", "O").replace("ø", "o").replace("Æ", "AE").replace("æ", "ae")
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.upper().replace("&", " AND ")


def specific_cleaning(name):
    "Brackets, wrappers, abbreviations, filler words, trailing noise, word order of universities."
    s = ascii_upper(name)
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = " ".join(s.split())
    for p in NAME_PREFIXES:
        s = re.sub(p, "", s)
    words = [ABBREVIATIONS.get(w, w) for w in s.split()]
    words = [w for w in words if w not in FILLER_WORDS]
    while words and words[-1] in TRAILING_NOISE:
        words.pop()
    if "UNIVERSITY" in words:                                  # OXFORD UNIVERSITY == UNIVERSITY OF OXFORD
        words = ["UNIVERSITY"] + [w for w in words if w != "UNIVERSITY"]
    return " ".join(words)


def my_harmonisation(name, use_aliases=True):
    "A (dictionary) + B (cleanco) + specific cleaning + manual aliases of the sector's big players."
    if name is None or str(name).strip() in ("", "nan", "None"):
        return ""
    base = clean_with_cleanco(ascii_upper(name))   # apply method B and then method A
    base = specific_cleaning(base)                 # add specific cleaning
    base = clean_with_dictionary(base)             # apply again method A
    base = strip_misspelled_legal_forms(base)      # 
    if use_aliases:
        for rx, canon in _ORG:
            if rx.search(base):
                return canon
    return base

In [21]:
ORG_WORDS = {"TECHNOLOGY", "SYSTEMS", "QUANTUM", "COMPUTING", "LABORATORY", "LABORATORIES", "SOLUTIONS",
             "SEMICONDUCTOR", "ELECTRONICS", "SOFTWARE", "DEVICES", "NETWORKS", "INDUSTRIES", "ENERGY",
             "RESEARCH", "SCIENCE", "COMMUNICATIONS", "MICRO", "DIGITAL", "DATA"}
ACADEMIC_WORDS = {"UNIVERSITY", "COLLEGE", "ACADEMY", "FACULTY", "SCHOOL", "POLYTECHNIC"}
GOV_WORDS = {"INSTITUTE", "NATIONAL", "CENTER", "MINISTRY", "COMMISSARIAT", "FOUNDATION", "COUNCIL",
             "GOVERNMENT", "LABORATORY", "DEPARTMENT"}


def applicant_type(psn_sector, raw_name=""):
    "Type from psn_sector; when it is empty, infer it from the name."
    s = str(psn_sector).upper()
    if "UNIVERSITY" in s or any(t in s for t in ["ACADEMY", "COLLEGE", "FACULTY"]): return "university"
    if "HOSPITAL" in s: return "hospital"
    if any(t in s for t in ["GOV", "MINISTRY", "LABORATORY", "PUBLIC", "RESEARCH INSTITUTE"]): return "government / non-profit"
    if "INDIVIDUAL" in s: return "individual"
    if "COMPANY" in s or "FIRM" in s: return "firm"
    # psn_sector missing: guess from the name
    words = set(re.sub(r"[^A-Z0-9 ]", " ", ascii_upper(raw_name)).split())
    if words & ACADEMIC_WORDS: return "university"
    if words & GOV_WORDS: return "government / non-profit"
    if words & (LEGAL_FORMS - {"THE", "CO"}) or words & ORG_WORDS: return "firm"
    if 2 <= len(words) <= 4: return "individual"      # "HOU XIAOYAN", "ROSE GEORDIE"
    return "unknown"

def build_name_map(raw, use_aliases=True, link=True):
    "{PATSTAT name: applicant}: A + B + specific cleaning + aliases, then subsidiaries linked to the brand."
    names = raw.dropna(subset=["psn_name"])
    names = names[names["psn_name"].str.strip() != ""]
    base_map = {n: my_harmonisation(n, use_aliases) for n in sorted(names["psn_name"].unique())}
    if not link:
        return base_map
    is_indiv = np.array([applicant_type(s, n) == "individual"
                         for s, n in zip(names["psn_sector"], names["psn_name"])])
    indiv = {base_map[n] for n in names.loc[is_indiv, "psn_name"].unique()} # individuals never absorb other names
    return link_subsidiaries(base_map, skip=indiv)


def consolidated_names(psn_names, use_aliases=True):
    "Dictionary {PATSTAT name: consolidated name}."
    return {n: my_harmonisation(n, use_aliases) for n in sorted(set(psn_names))}


def prepare_applicants(raw, use_aliases=True, link=True):
    "From (unit_id, year, person) rows to one row per (unit_id, applicant) with type and country."
    df = raw.copy()
    df = df.dropna(subset=["psn_name"])
    df = df[df["psn_name"].str.strip() != ""]

    # 1. consolidated name
    df["applicant"] = df["psn_name"].map(build_name_map(df, use_aliases, link))
    df = df[df["applicant"].str.strip() != ""]

    # 2. type: from psn_sector or from the name, then the most frequent type of each applicant
    df["type"] = [applicant_type(s, n) for s, n in zip(df["psn_sector"], df["psn_name"])]
    known_t = df[df["type"] != "unknown"]
    if not known_t.empty:
        main_type = known_t.groupby("applicant")["type"].agg(lambda t: t.value_counts().index[0])
        df["type"] = df["applicant"].map(main_type).fillna("unknown")

    # 3. one country per applicant: the most frequent known one
    df["ctry"] = df["ctry"].replace("", np.nan)
    known = df.dropna(subset=["ctry"]).drop_duplicates(["unit_id", "applicant", "ctry"])
    if not known.empty:
        main_c = known.groupby("applicant")["ctry"].agg(lambda c: c.value_counts().index[0])
        df["ctry"] = df["applicant"].map(main_c)

    # 4. one row per (patent, applicant)
    return df[["unit_id", "year", "applicant", "type", "ctry"]].drop_duplicates(["unit_id", "applicant"])

Now, we can apply the harmonisation directly on our applicants: we adapt method A and B to our own sector, adding some specific harmonisation.

In [22]:
df_applicants = prepare_applicants(raw_sector)

print(f"Total rows after cleaning and deduplication: {len(df_applicants)}")
print(f"Unique harmonised applicants: {df_applicants['applicant'].nunique()}")
print(f"Raw PATSTAT names: {raw_sector['psn_name'].nunique():,}")
print(df_applicants["type"].value_counts())
print("-" * 50)

df_applicants.head(15)

Total rows after cleaning and deduplication: 17057
Unique harmonised applicants: 4347
Raw PATSTAT names: 5,384
type
firm                       9173
university                 3490
individual                 3303
government / non-profit    1008
unknown                      83
Name: count, dtype: int64
--------------------------------------------------


,unit_id,year,applicant,type,ctry
1,68480559,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
2,68480760,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
3,68480568,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
4,75188684,2020,INFORMATION SCIENCE ACADEMY CHINA,university,NaN
5,68480589,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
6,68480607,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
7,68480742,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
8,50883643,2012,KOREA RESEARCH INSTITUTE STANDARD SCIENCE,government / non-profit,KR
9,34954098,2004,CNRS,government / non-profit,FR
10,54063171,2015,UNIVERSITY OXFORD,university,GB


### 2.4 Top-N audit for Quantum Computing by number of families

In [23]:
# Audit: top applicants by number of families, with the PATSTAT names merged into each one
name_map = build_name_map(raw_sector)
top = df_applicants.groupby("applicant")["unit_id"].nunique().sort_values(ascending=False).head(TOP_N)
variants = (raw_sector.dropna(subset=["psn_name"]).assign(applicant=lambda d: d["psn_name"].map(name_map))
            .groupby("applicant")["psn_name"].agg(lambda x: " | ".join(sorted(set(x)))[:200]))
audit = top.rename("families").to_frame().join(variants.rename("merged names"))
audit

,families,merged names
applicant,,
IBM,598,IBM (CHINA) INVESTMENT COMPANY | IBM (INTERNAT...
ORIGIN QUANTUM,560,"ORIGIN QUANTUM COMPUTING COMPANY, LIMITED, HEF..."
MICROSOFT,214,MICROSOFT CORPORATION | MICROSOFT LICENSING TE...
QUANTUMCTEK,198,QUANTUMCTEK (BEIJING) COMPANY | QUANTUMCTEK (G...
D-WAVE SYSTEMS,198,D-WAVE (COMMERCIAL) | D-WAVE SYSTEMS | D-WAVE ...
TOSHIBA,180,TOSHIBA CORPORATION | TOSHIBA DIGITAL SOLUTION...
BAIDU,179,BEIJING BAIDU NETCOM SCIENCE AND TECHNOLOGY CO...
GOOGLE,176,GOOGLE | GOOGLE LIMITED RESPONSIBILITY COMPANY...
INTEL,153,INTEL CORPORATION


In [24]:
before = pd.Series(build_name_map(raw_sector, link=False))
after = pd.Series(name_map)
linked = (pd.DataFrame({"before": before, "after": after}).query("before != after")
          .groupby("after")["before"]
          .agg(n_absorbed="nunique", examples=lambda x: " | ".join(sorted(set(x))[:6])))
linked.sort_values("n_absorbed", ascending=False).head(40)

,n_absorbed,examples
after,,
STATE GRID,39,STATE GRID ANHUI ELECTRIC POWER CO LTD INFORMA...
ALIBABA,8,ALIBABA CHINA | ALIBABA CLOUD COMPUTING | ALIB...
TENCENT,6,TENCENT AMERICA | TENCENT CLOUD EUROPE GERMANY...
FUJITSU,4,FUJITSU GERMANY | FUJITSU SERVICES | FUJITSU T...
CHINA TELECOM,3,CHINA TELECOM DIGITAL INTELLIGENCE TECHNOLOGY ...
INSPUR,3,INSPUR CLOUD INFORMATION TECHNOLOGY | INSPUR S...
FRAUNHOFER,3,FRAUNHOFER GESELLSCHAFT ZUR FORDERUNG ANGEWAND...
NOKIA,3,NOKIA NETWORKS | NOKIA TECHNOLOGY | NOKIA USA
HITACHI,3,HITACHI EUROPE | HITACHI INFORMATION COMMUNICA...


### 2.5 Test our method: manual check of 100 names

We validate the harmonisation on our own data with two sheets of 50 judgements each:

- **Sheet 1, precision**: are the PATSTAT names merged into the same applicant really the same organisation? We sample 50 applicants that merge at least two names: the 30 with most families and 20 at random from the rest.
- **Sheet 2, missed merges**: are two look-alike applicants that were *not* merged really the same organisation? We sample 50 such pairs among the top-300 applicants.

For each row we fill `same_org` with `yes` or `no` (when the merge is a group-level choice, e.g. *IBM (China)* inside *IBM*, we write `yes` and note it).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

fam = df_applicants.groupby("applicant")["unit_id"].nunique().rename("families")
variants = (raw_sector.dropna(subset=["psn_name"]).assign(applicant=lambda d: d["psn_name"].map(name_map))
            .groupby("applicant")["psn_name"].agg(lambda x: sorted(set(x))))
aud = pd.concat([fam, variants.rename("psn_names")], axis=1).dropna()
aud["n_names"] = aud["psn_names"].map(len)

# SHEET 1 - PRECISION: 30 biggest + 20 random, among the applicants that merge >= 2 PATSTAT names
merged = aud[aud["n_names"] >= 2].sort_values("families", ascending=False)
sheet_precision = pd.concat([merged.head(30), merged.iloc[30:].sample(n=min(20, len(merged) - 30), random_state=42)])
sheet_precision = sheet_precision.assign(psn_names=sheet_precision["psn_names"].map(" | ".join), same_org="")

# SHEET 2 - MISSED MERGES: look-alike pairs that were NOT merged, among the top-300 applicants
names_all = fam.index.tolist()
X = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(names_all)
top_idx = np.argsort(-fam.values)[:300]
S = (X[top_idx] @ X.T).toarray()
rows = []
for r, i in enumerate(top_idx):
    S[r, i] = 0
    for j in np.argsort(-S[r])[:3]:
        if S[r, j] >= 0.6:
            rows.append((names_all[i], int(fam.iloc[i]), names_all[j], int(fam.iloc[j]), round(float(S[r, j]), 2)))
cands = pd.DataFrame(rows, columns=["applicant_1", "families_1", "applicant_2", "families_2", "similarity"])
cands["pair"] = [tuple(sorted(p)) for p in zip(cands["applicant_1"], cands["applicant_2"])]
sheet_miss = (cands.drop_duplicates("pair").drop(columns="pair")
              .sample(n=50, random_state=42).assign(same_org=""))

sheet_precision.to_csv(OUTPUT_DIR / "audit_sheet_precision.csv")
sheet_miss.to_csv(OUTPUT_DIR / "audit_sheet_missed_merges.csv", index=False)

Compili la colonna same_org con yes o no (se unite per scelta di gruppo, come IBM China dentro IBM, scrivi yes e annotalo). Poi:

precision = quota di yes nel foglio 1;
merge mancati = quota di yes nel foglio 2.

Per il technical report, scrivi le due percentuali con l’intervallo di Wilson (c’è in src/validation.py) e il numero di giudizi. Se sheet_miss ha meno di 50 righe perché le coppie simili sono poche, prendi tutte quelle che ci sono: è un buon segno.

**Reading the sheets.** Precision = share of `yes` in sheet 1. Missed merges = share of `yes` in sheet 2. We report both with a Wilson confidence interval and the number of judgements.